# Notebook 02 — Baseline Model (seeing the problem)

**Goal:** Train a naive Random Forest and see how badly class imbalance hurts it.

After this notebook you will understand:
- Why accuracy is a misleading metric for imbalanced data
- How to read a confusion matrix
- How to read a classification report (precision, recall, F1)
- Why the minority classes (Critically Endangered, Extinct) get terrible scores

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd

from src.load_data import load_processed, LABEL_COL
from src.preprocess import get_X_y, ENDANGERMENT_ORDER
from src.train import split_data, train_baseline, save_model
from src.evaluate import evaluate_model, print_classification_report
from src.visualise import plot_confusion_matrix, plot_feature_importance

print('Imports OK')

In [ ]:
# ── Load cleaned data ─────────────────────────────────────────────────
# (Run notebook 01 first if this fails)
df = load_processed()

In [ ]:
# ── Extract features and labels ───────────────────────────────────────
# get_X_y() returns:
#   X — numeric feature matrix (one-hot encoded)
#   y — label series (string class names)
X, y = get_X_y(df)
print(f'X shape: {X.shape}')
print(f'Feature names: {list(X.columns[:5])} ...')

In [ ]:
# ── Train/test split ──────────────────────────────────────────────────
# IMPORTANT: we use stratify=y so rare classes appear in both splits.
# Open src/train.py and read split_data() to see how this works.
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)

In [ ]:
# ── Train baseline model ──────────────────────────────────────────────
# No special imbalance handling — this is deliberately naive.
# We want to SEE the problem before fixing it.
model_base = train_baseline(X_train, y_train)
save_model(model_base, 'baseline')

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────
results_base = evaluate_model(model_base, X_test, y_test, ENDANGERMENT_ORDER)
print_classification_report(results_base)

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────
# Read ABOUT_THE_PROJECT.md for a detailed explanation of confusion matrices.
#
# What to look for:
#   - Are the DIAGONAL values high? (good = model correct)
#   - Are the OFF-DIAGONAL values high? (bad = mistakes)
#   - Are the last rows (Critically Endangered, Extinct) mostly zeros? (problem!)

plot_confusion_matrix(
    results_base['cm'],
    results_base['cm_labels'],
    title='Baseline model — confusion matrix',
    normalise=False,
    save_name='02_cm_baseline'
)

In [ ]:
# ── Normalised confusion matrix ───────────────────────────────────────
# Same matrix but each row divided by true class count.
# Shows RECALL per class as a percentage.
# Makes it easy to see which classes the model is good/bad at.
plot_confusion_matrix(
    results_base['cm'],
    results_base['cm_labels'],
    title='Baseline model — normalised confusion matrix (recall per class)',
    normalise=True,
    save_name='02_cm_baseline_normalised'
)

In [ ]:
# ── Feature importance ────────────────────────────────────────────────
# Which features did the model actually use to make decisions?
# Number of speakers should dominate — but let's verify.
plot_feature_importance(
    model_base,
    list(X.columns),
    top_n=15,
    save_name='02_feature_importance'
)

In [ ]:
# ── Key takeaway ──────────────────────────────────────────────────────
print('=== KEY TAKEAWAY FROM NOTEBOOK 02 ===')
print(f'Overall accuracy: {results_base["accuracy"]:.1%}  (looks OK but is misleading!)')
print(f'Macro F1:         {results_base["macro_f1"]:.3f}  (this is the real score)')
print()
print('Look at the per-class F1 in the report above.')
print('Critically Endangered and Extinct likely have F1 close to 0.')
print('The model learned to predict the majority class.')
print()
print('Proceed to 03_imbalance_fix.ipynb to fix this.')